# miRanda Target Scan

![miRanda Target Scan](https://proto-bio.github.io/proto-assets/images/tool/miranda/hero.png)

This notebook demonstrates `run_miranda_scan`, which predicts microRNA target sites in RNA/DNA sequences by combining Smith-Waterman complementarity alignment with ViennaRNA duplex thermodynamics. The example reproduces the canonical *Drosophila* miR-bantam / *hid* 3'UTR interaction. See the miRanda paper for background: [Enright et al., Genome Biology (2003)](https://doi.org/10.1186/gb-2003-5-1-r1).

In [ ]:
from proto_tools.tools.gene_annotation.miranda import run_miranda_scan, MirandaInput, MirandaConfig
from proto_tools.utils.notebook_docs import display_api_reference

## Input

`MirandaInput` takes one or more target sequences to scan. The microRNA query set lives on `MirandaConfig` (`mirna_queries`) because the same queries apply to every target in a scan, so they're a shared configuration rather than per-target data.

In [ ]:
display_api_reference("miranda-scan", "input", "run_miranda_scan")

## Config

`MirandaConfig` controls scoring thresholds and alignment penalties. Defaults mirror the miRanda CLI; lower `score_threshold` or raise `energy_threshold` toward 0 for higher sensitivity.

In [ ]:
display_api_reference("miranda-scan", "config", "run_miranda_scan")

## Output

`MirandaOutput` returns one `MirandaSequenceResult` per target sequence, each holding the predicted `target_sites` sorted by score descending.

In [ ]:
display_api_reference("miranda-scan", "output", "run_miranda_scan")

## Basic usage

Scan the real *Drosophila* miR-bantam microRNA against the *hid* 3'UTR. The miR-bantam sequence is short, so we paste it inline; the *hid* 3'UTR is long, so we read it from the bundled `hid_utr.fasta` next to this notebook.

In [ ]:
from pathlib import Path


def read_fasta_sequence(path):
    """Return the concatenated sequence from a single-record FASTA file."""
    lines = Path(path).read_text().splitlines()
    return "".join(line.strip() for line in lines if not line.startswith(">"))


# miR-bantam microRNA query (real sequence) and the hid 3'UTR target.
mir_bantam = "gtgagatcattttgaaagctg"
hid_utr = read_fasta_sequence("hid_utr.fasta")

inputs = MirandaInput(target_sequences=[hid_utr])
config = MirandaConfig(mirna_queries=[mir_bantam], mirna_ids=["miR-bantam"])

# Run the scan with default scoring thresholds.
result = run_miranda_scan(inputs, config)

# Results come back one per input target, in order. Inspect the hid 3'UTR hits.
hid_result = result.results[0]
print(f"hid 3'UTR: {hid_result.num_sites} target site(s) found")

top_site = hid_result.target_sites[0]
print(f"  microRNA:     {top_site.mirna_id}")
print(f"  score:        {top_site.score}")
print(f"  energy:       {top_site.energy} kcal/mol")
print(f"  target_range: {top_site.target_start}-{top_site.target_end} (1-indexed, inclusive)")

## Advanced usage

Loosen the search with a non-default config. Raising `energy_threshold` toward 0 admits weaker duplexes. This finds more candidate sites at the cost of more false positives.

In [ ]:
# More sensitive: a looser energy cutoff.
sensitive_config = MirandaConfig(
    mirna_queries=[mir_bantam], mirna_ids=["miR-bantam"], energy_threshold=-14
)
sensitive_result = run_miranda_scan(inputs, sensitive_config)

default_sites = result.results[0].num_sites
sensitive_sites = sensitive_result.results[0].num_sites
print(f"default config:   {default_sites} site(s)")
print(f"sensitive config: {sensitive_sites} site(s)")
print(f"additional sites found: {sensitive_sites - default_sites}")

## Export results

Predicted sites can be exported to `csv` or `json`; both are one flat row per target site. Here we write the default-config results to CSV and print the table head.

In [ ]:
import pandas as pd

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Supported formats: result.output_format_options -> ['csv', 'json'].
result.export(name="miranda_sites", export_path=output_dir, file_format="csv")

print(pd.read_csv(output_dir / "miranda_sites.csv").head())